# EditJudge on Kaggle (template)

Copy this notebook for a task. Before running:
- add the private Kaggle dataset `editjudge-data` as input (made from `python -m src.process pack`; share it only within the team: it contains photos of real people);
- notebook settings: GPU accelerator on, internet on (to clone the code and download model weights);
- push the code you want to run to GitHub first: the notebook clones `BRANCH`;
- get the coordinator's approval for the run.

Outputs are written to `/kaggle/working/out` and kept as the notebook's output; never commit them.

In [ ]:
import os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "curr"                            # or your task branch, e.g. "instructions/hai"
DATA = "/kaggle/input/editjudge-data"      # the private dataset (editjudge-data.zip, extracted by Kaggle)
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

os.environ["EJ_DATA_DIR"] = DATA           # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT          # all results are written here
os.makedirs(OUT, exist_ok=True)
os.chdir(CODE)

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:                                       # optional Kaggle secret HF_TOKEN (Add-ons > Secrets)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:                          # public models need no token: hide the rate-limit notice
    os.environ["HF_HUB_VERBOSITY"] = "error"

In [ ]:
# Install only what Kaggle lacks (check with pip list first)
!pip install -q onnxruntime open_clip_torch

In [ ]:
# Smoke test on a few images, then the full run
!python -m src.filters list
!python -m src.filters run --task nsfw --model MODEL_NAME --limit 20
# !python -m src.filters run --task nsfw --model MODEL_NAME

In [ ]:
!ls -R /kaggle/working/out | head